In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

DATA PREPARATION!

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# dataset
df = pd.read_csv("fmnist_small.csv")

# split train and test
X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

TRANSFORMING THE DATA!

In [5]:
from torchvision.transforms import transforms

# the following seetings are for vgg16
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

DATA LOADING!

In [6]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train, y_train, transforms):
        self.X_train_numpy = X_train
        self.y_train_numpy = y_train
        self.transform = transforms
    def __len__(self):
        return len(self.X_train_numpy)
    def __getitem__(self,idx):
        # resize to (28, 28)
        image = self.X_train_numpy[idx].reshape(28,28)
        # change datatype to np.uint8
        image = image.astype(np.uint8)
        # change black&white to color -> (H,W,C) -> (C,H,W)
        image = np.stack([image]*3, axis=-1)
        # convert array to PIL image
        image = Image.fromarray(image)
        # apply transforms to convert the image into tensors!
        image_tensor = self.transform(image)
        # converting labels to tensors!
        labels_tensor = torch.tensor(self.y_train_numpy[idx], dtype=torch.long)
        # return
        return image_tensor, labels_tensor
    
train_dataset = CustomDataset(X_train.values, y_train.values, custom_transform)
test_dataset = CustomDataset(X_test.values, y_test.values, custom_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory=True)

BUILT-IN MODEL INITIALIZING!

In [7]:
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

/home/sriram/Documents/myenv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/sriram/Documents/myenv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FREEZING THE FEATURE EXTRACTOR!

In [8]:
for param in vgg16.features.parameters():
    param.requires_grad=False

TRAIN THE CLASSIFIER ARCHITECTURE!

In [9]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

INITIALIZE THE PARAMETERS!

In [10]:
from torch import optim

# set learning rate and epochs
epochs = 50
learning_rate = 0.1
lambda_val = 1e-4

# vgg to GPU
vgg16.to(device)
# loss function
loss_fun = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.SGD(vgg16.parameters(), lr=learning_rate, weight_decay=lambda_val)

TRAINING LOOP!

In [11]:
# training loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    # zero the prev grads
    optimizer.zero_grad()
    # move the tensor data to GPU!
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    # forward pass
    outputs = vgg16(batch_features)
    # calculate loss
    loss = loss_fun(outputs, batch_labels)
    # back pass
    loss.backward()
    # update grads
    optimizer.step()
    # calculate loss for all items in a batch
    total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 0.9336099664370219
Epoch: 2 , Loss: 0.49840977842609085
Epoch: 3 , Loss: 0.4043249929447969
Epoch: 4 , Loss: 0.32817584936817484
Epoch: 5 , Loss: 0.2957393138110638
Epoch: 6 , Loss: 0.2659532754619916
Epoch: 7 , Loss: 0.22624289124505595
Epoch: 8 , Loss: 0.18887174524987738
Epoch: 9 , Loss: 0.1674491531526049
Epoch: 10 , Loss: 0.14237161304801702
Epoch: 11 , Loss: 0.12997706495846312
Epoch: 12 , Loss: 0.12126287143056591
Epoch: 13 , Loss: 0.0950111923723792
Epoch: 14 , Loss: 0.11460846726006518
Epoch: 15 , Loss: 0.07733758530889948
Epoch: 16 , Loss: 0.07863695635149877
Epoch: 17 , Loss: 0.09195209349127254
Epoch: 18 , Loss: 0.07949963560754744
Epoch: 19 , Loss: 0.06527233515875802
Epoch: 20 , Loss: 0.0652675367690002
Epoch: 21 , Loss: 0.0679653162084287
Epoch: 22 , Loss: 0.04360225496556571
Epoch: 23 , Loss: 0.0570676713442117
Epoch: 24 , Loss: 0.06339814471857001
Epoch: 25 , Loss: 0.06386177917399133
Epoch: 26 , Loss: 0.043215417006528395
Epoch: 27 , Loss: 0.035617110

EVALUATION LOOP ON TEST DATA!

In [12]:
# set model to eval mode
vgg16.eval()

# evaluation code
total = len(y_test)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = vgg16(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8941666666666667


EVALUATION LOOP ON TRAIN DATA!

In [13]:
# set model to eval mode
vgg16.eval()

# evaluation code
total = len(y_train)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = vgg16(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9989583333333333
